In [13]:
import json
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import datetime as dt


def save_export_link_to_source(source_rec_path, dest_full_path):
    """
    Save a link file in the source folder pointing to where it was exported.
    Skips if already exists.
    """
    link_file = os.path.join(source_rec_path, "exported_to.json")
    
    if os.path.exists(link_file):
        print(f"⏭️  Skip (exists): {link_file}")
        return
    
    data = {
        "dest_path": dest_full_path
    }
    
    with open(link_file, 'w') as f:
        json.dump(data, f, indent=2)
    
    print(f"✅ Saved export link: {link_file}")


def add_dest_to_folder_log(dest_folder, dest_full_path):
    """
    Add export destination path to the folder_log.parquet in metadata.
    Skips if column already exists.
    """
    metadata_dir = os.path.join(dest_folder, "metadata")
    log_path = os.path.join(metadata_dir, "folder_log.parquet")
    
    if not os.path.exists(log_path):
        print(f"⚠️  No folder_log.parquet found in {metadata_dir}")
        return
    
    # Read existing parquet
    table = pq.read_table(log_path)
    df = table.to_pandas()
    
    # Skip if column already exists
    if 'export_dest_path' in df.columns:
        print(f"⏭️  Skip (column exists): {log_path}")
        return
    
    # Add new column
    df['export_dest_path'] = dest_full_path
    
    # Convert back to Arrow table and save
    new_table = pa.Table.from_pandas(df)
    pq.write_table(new_table, log_path)
    
    print(f"✅ Added dest_path to: {log_path}")

In [ ]:
# Test 1: Save link in source
source = "/data/big_rim/rsync_dcc_sum/Oct3V1/2024_11_06/20241015pmcr2_16_53"
dest = "/data/big_rim/sorted_mir_dataset/oct3v1_single_mini_good/single/20241106_MC7_S1"

save_export_link_to_source(source, dest)

# Test 2: Update folder_log
add_dest_to_folder_log(dest, dest)

⏭️  Skip (exists): /data/big_rim/rsync_dcc_sum/Oct3V1/2024_11_06/20241015pmcr2_16_53/exported_to.json
⏭️  Skip (column exists): /data/big_rim/sorted_mir_dataset/oct3v1_single_mini_good/single/20241106_MC7_S1/metadata/folder_log.parquet


In [24]:
import os
import json
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa

def save_export_link_to_source(source_rec_path, dest_full_path, dest_rel_path, skip_if_exists=True):
    """
    Save link in source folder with BOTH absolute and relative dest paths.
    
    Args:
        source_rec_path: Absolute path to source
        dest_full_path: Absolute path to destination
        dest_rel_path: RELATIVE path including dataset folder (e.g., "oct3v1_single_mini/single/20241106_MC7_S1/")
        skip_if_exists: Skip if file already exists
    """
    link_file = os.path.join(source_rec_path, "exported_to.json")
    
    if skip_if_exists and os.path.exists(link_file):
        return False
    
    data = {
        "dest_full_path": dest_full_path,
        "dest_rel_path": dest_rel_path
    }
    
    with open(link_file, 'w') as f:
        json.dump(data, f, indent=2)
    
    return True

def add_dest_to_folder_log(dest_folder, dest_full_path, dest_rel_path, skip_if_exists=True):
    """
    Add BOTH absolute and relative dest paths to folder_log.parquet.
    
    Args:
        dest_folder: Full path to exported folder
        dest_full_path: Absolute path to destination
        dest_rel_path: RELATIVE path including dataset folder
        skip_if_exists: Skip if columns already exist
    """
    metadata_dir = os.path.join(dest_folder, "metadata")
    log_path = os.path.join(metadata_dir, "folder_log.parquet")
    
    if not os.path.exists(log_path):
        return False
    
    table = pq.read_table(log_path)
    df = table.to_pandas()
    
    # Check if EITHER column exists
    has_columns = 'export_dest_full_path' in df.columns or 'export_dest_rel_path' in df.columns
    
    if skip_if_exists and has_columns:
        return False
    
    # Add BOTH columns
    df['export_dest_full_path'] = dest_full_path
    df['export_dest_rel_path'] = dest_rel_path
    
    new_table = pa.Table.from_pandas(df)
    pq.write_table(new_table, log_path)
    
    return True

def process_all_exported_folders(export_root, skip_if_exists=True):
    """
    Walk through exported dataset and add BOTH absolute and relative path mappings.
    Relative path now includes the dataset folder name.
    Set skip_if_exists=False to overwrite existing mappings.
    """
    stats = {
        'processed': 0,
        'source_links_added': 0,
        'dest_columns_added': 0,
        'skipped_no_rec_path': 0,
        'errors': []
    }
    
    # Get dataset folder name (e.g., "oct3v1_single_mini_good")
    dataset_name = os.path.basename(export_root)
    
    print(f"Processing: {export_root}")
    print(f"Dataset name: {dataset_name}")
    print(f"Skip if exists: {skip_if_exists}")
    print("="*80)
    
    for condition in os.listdir(export_root):
        condition_path = os.path.join(export_root, condition)
        if not os.path.isdir(condition_path):
            continue
        
        print(f"\n📁 Condition: {condition}")
        
        for session_folder in os.listdir(condition_path):
            session_path = os.path.join(condition_path, session_folder)
            if not os.path.isdir(session_path):
                continue
            
            stats['processed'] += 1
            
            # Read source rec_path from folder_log.parquet
            log_path = os.path.join(session_path, "metadata", "folder_log.parquet")
            
            if not os.path.exists(log_path):
                stats['skipped_no_rec_path'] += 1
                print(f"  ⏭️  Skip (no folder_log): {session_folder}")
                continue
            
            try:
                df = pd.read_parquet(log_path)
                if 'rec_path' not in df.columns:
                    stats['skipped_no_rec_path'] += 1
                    print(f"  ⏭️  Skip (no rec_path): {session_folder}")
                    continue
                
                source_rec_path = df['rec_path'].iloc[0]
                dest_full_path = session_path
                
                # Calculate RELATIVE dest path INCLUDING dataset folder name
                # e.g., "oct3v1_single_mini_good/single/20241106_MC7_S1/"
                dest_rel_path = os.path.join(dataset_name, condition, session_folder) + "/"
                
                # 1. Add link to source folder (with BOTH paths)
                if save_export_link_to_source(source_rec_path, dest_full_path, dest_rel_path, skip_if_exists):
                    stats['source_links_added'] += 1
                    print(f"  ✅ Added source link: {session_folder}")
                else:
                    print(f"  ⏭️  Source link exists: {session_folder}")
                
                # 2. Add BOTH paths to folder_log
                if add_dest_to_folder_log(session_path, dest_full_path, dest_rel_path, skip_if_exists):
                    stats['dest_columns_added'] += 1
                    print(f"  ✅ Added dest columns: {session_folder}")
                else:
                    print(f"  ⏭️  Dest columns exist: {session_folder}")
                    
            except Exception as e:
                stats['errors'].append((session_folder, str(e)))
                print(f"  ❌ Error: {session_folder} - {e}")
    
    return stats

# ============================================================================
# RUN - Set skip_if_exists=False to UPDATE ALL
# ============================================================================

DATASETS = [
    "/data/big_rim/uploade_ssh_mir_dataset/oct3v1_beh_only",
    "/data/big_rim/uploade_ssh_mir_dataset/oct3v1_social_mini"
    # "/data/big_rim/uploade_ssh_mir_dataset/oct3v1_single_beh",
    # "/data/big_rim/uploade_ssh_mir_dataset/oct3v1_single_mini_good",
    # "/data/big_rim/uploade_ssh_mir_dataset/oct3v1_social_bev",
    # "/data/big_rim/uploade_ssh_mir_dataset/oct3v1_social_mini"
]

SKIP_EXISTING = False  # Set to False to UPDATE ALL with new relative paths

print("="*80)
print("BATCH PROCESSING: Adding BOTH absolute and relative path mappings")
print(f"Mode: {'UPDATE ALL' if not SKIP_EXISTING else 'SKIP EXISTING'}")
print("="*80)

all_stats = []
for dataset in DATASETS:
    if os.path.exists(dataset):
        stats = process_all_exported_folders(dataset, skip_if_exists=SKIP_EXISTING)
        all_stats.append((dataset, stats))
    else:
        print(f"\n⚠️  Dataset not found: {dataset}")

# Print summary
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
for dataset, stats in all_stats:
    print(f"\n📁 {os.path.basename(dataset)}")
    print(f"   Processed: {stats['processed']} folders")
    print(f"   Source links added/updated: {stats['source_links_added']}")
    print(f"   Dest columns added/updated: {stats['dest_columns_added']}")
    print(f"   Errors: {len(stats['errors'])}")

print("\n✅ Done!")
print("\nNow relative paths include dataset folder:")
print("  e.g., 'oct3v1_single_mini_good/single/20241106_MC7_S1/'")

BATCH PROCESSING: Adding BOTH absolute and relative path mappings
Mode: UPDATE ALL
Processing: /data/big_rim/uploade_ssh_mir_dataset/oct3v1_beh_only
Dataset name: oct3v1_beh_only
Skip if exists: False

📁 Condition: .trash
  ⏭️  Skip (no folder_log): 20251028_132918

📁 Condition: single
  ✅ Added source link: 20241113_VC3_S1
  ✅ Added dest columns: 20241113_VC3_S1
  ✅ Added source link: 20241004_VC4_S5
  ✅ Added dest columns: 20241004_VC4_S5
  ✅ Added source link: 20240918_MC14_S1
  ✅ Added dest columns: 20240918_MC14_S1
  ✅ Added source link: 20241106_MC17_S1
  ✅ Added dest columns: 20241106_MC17_S1
  ✅ Added source link: 20241007_VC10_S3
  ✅ Added dest columns: 20241007_VC10_S3
  ✅ Added source link: 20250521_VC12_S1
  ✅ Added dest columns: 20250521_VC12_S1
  ✅ Added source link: 20250213_MC16_S2
  ✅ Added dest columns: 20250213_MC16_S2
  ✅ Added source link: 20250514_MC12_S2
  ✅ Added dest columns: 20250514_MC12_S2
  ✅ Added source link: 20241008_VC10_S7
  ✅ Added dest columns: 20241

In [2]:
import os
import json
import pandas as pd

def build_mapping_table_from_logs(export_log_paths, output_csv="export_mapping_master.csv"):
    """
    Read all export logs and build a master mapping table.
    Append-only: won't overwrite existing mappings.
    
    Args:
        export_log_paths: List of paths to export_log_*.parquet files
        output_csv: Where to save the master table
    """
    
    # Load existing mappings if they exist
    if os.path.exists(output_csv):
        print(f"📖 Loading existing mappings from: {output_csv}")
        existing_df = pd.read_csv(output_csv)
        existing_sources = set(existing_df['source_rec_path'].values)
        print(f"   Found {len(existing_sources)} existing mappings")
    else:
        existing_df = pd.DataFrame()
        existing_sources = set()
        print("📝 Creating new mapping table")
    
    # Collect all mappings from logs
    all_mappings = []
    
    for log_path in export_log_paths:
        if not os.path.exists(log_path):
            print(f"⚠️  Log not found: {log_path}")
            continue
        
        print(f"\n📂 Reading: {log_path}")
        log_df = pd.read_parquet(log_path)
        
        new_count = 0
        for _, row in log_df.iterrows():
            source = row['source_rec_path']
            
            # Skip if already in existing mappings
            if source in existing_sources:
                continue
            
            all_mappings.append({
                'source_rec_path': source,
                'dest_rel_path': row['dest_rel_path'],
                'animal_id': row['id'],
                'is_social': row['is_social'],
                'export_time': row['export_time_utc'],
                'profile_label': row.get('profile_label', ''),
                'dataset_version': row.get('dataset_version', ''),
            })
            new_count += 1
        
        print(f"   Added {new_count} new mappings")
    
    # Combine with existing
    if len(all_mappings) > 0:
        new_df = pd.DataFrame(all_mappings)
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
        
        # Save
        combined_df.to_csv(output_csv, index=False)
        print(f"\n✅ Saved master table: {output_csv}")
        print(f"   Total mappings: {len(combined_df)}")
    else:
        print(f"\n⏭️  No new mappings to add")
        combined_df = existing_df
    
    return combined_df

# ============================================================================
# RUN
# ============================================================================

LOG_PATHS = [
    "/home/lq53/mir_repos/BBOP/random_tests/25oct_dataset_prep/export_logs/export_log_oct3v1_single_beh.parquet",
    "/home/lq53/mir_repos/BBOP/random_tests/25oct_dataset_prep/export_logs/export_log_oct3v1_single_mini_good.parquet",
    "/home/lq53/mir_repos/BBOP/random_tests/25oct_dataset_prep/export_logs/export_log_oct3v1_social_bev.parquet",
    "/home/lq53/mir_repos/BBOP/random_tests/25oct_dataset_prep/export_logs/export_log_oct3v1_social_mini.parquet"
]

OUTPUT_CSV = "export_mapping_master.csv"

print("="*80)
print("BUILDING MASTER MAPPING TABLE")
print("="*80)

df = build_mapping_table_from_logs(LOG_PATHS, OUTPUT_CSV)

print("\n" + "="*80)
print("PREVIEW:")
print(df.head(10).to_string())
print("="*80)

BUILDING MASTER MAPPING TABLE
📝 Creating new mapping table

📂 Reading: /home/lq53/mir_repos/BBOP/random_tests/25oct_dataset_prep/export_logs/export_log_oct3v1_single_beh.parquet
   Added 47 new mappings

📂 Reading: /home/lq53/mir_repos/BBOP/random_tests/25oct_dataset_prep/export_logs/export_log_oct3v1_single_mini_good.parquet
   Added 38 new mappings

📂 Reading: /home/lq53/mir_repos/BBOP/random_tests/25oct_dataset_prep/export_logs/export_log_oct3v1_social_bev.parquet
   Added 25 new mappings

📂 Reading: /home/lq53/mir_repos/BBOP/random_tests/25oct_dataset_prep/export_logs/export_log_oct3v1_social_mini.parquet
   Added 33 new mappings

✅ Saved master table: export_mapping_master.csv
   Total mappings: 143

PREVIEW:
                                                    source_rec_path             dest_rel_path animal_id  is_social          export_time profile_label dataset_version
0    /data/big_rim/rsync_dcc_sum/Oct3V1/2025_05_21/20241217v1l23re1  single/20250521_VC12_S1/      VC12       

In [8]:
import os
import json
import pandas as pd
import pyarrow.parquet as pq

def validate_bidirectional_mapping(export_root, master_csv_path):
    """
    Validate that source and dest folders point to each other correctly.
    Updated to handle dest_full_path and dest_rel_path.
    """
    
    print(f"\n{'='*80}")
    print(f"VALIDATING: {export_root}")
    print(f"{'='*80}")
    
    # Load master CSV for comparison
    if os.path.exists(master_csv_path):
        master_df = pd.read_csv(master_csv_path)
        master_mappings = dict(zip(master_df['source_rec_path'], master_df['dest_rel_path']))
        print(f"📖 Loaded master CSV with {len(master_mappings)} mappings")
    else:
        master_mappings = {}
        print(f"⚠️  Master CSV not found: {master_csv_path}")
    
    stats = {
        'total_folders': 0,
        'valid_bidirectional': 0,
        'missing_source_link': 0,
        'missing_dest_column': 0,
        'source_link_mismatch': 0,
        'dest_column_mismatch': 0,
        'master_csv_match': 0,
        'master_csv_mismatch': 0,
        'not_in_master': 0,
        'errors': []
    }
    
    issues = {
        'missing_source_link': [],
        'missing_dest_column': [],
        'source_mismatch': [],
        'dest_mismatch': [],
        'master_mismatch': [],
        'not_in_master': []
    }
    
    # Walk through exported folders
    for condition in os.listdir(export_root):
        condition_path = os.path.join(export_root, condition)
        if not os.path.isdir(condition_path):
            continue
        
        for session_folder in os.listdir(condition_path):
            session_path = os.path.join(condition_path, session_folder)
            if not os.path.isdir(session_path):
                continue
            
            stats['total_folders'] += 1
            
            # Get paths
            metadata_dir = os.path.join(session_path, "metadata")
            log_path = os.path.join(metadata_dir, "folder_log.parquet")
            
            if not os.path.exists(log_path):
                stats['errors'].append((session_folder, "No folder_log.parquet"))
                continue
            
            try:
                # Read folder_log.parquet
                df = pd.read_parquet(log_path)
                
                if 'rec_path' not in df.columns:
                    stats['errors'].append((session_folder, "No rec_path in folder_log"))
                    continue
                
                source_rec_path = df['rec_path'].iloc[0]
                dest_full_path = session_path
                dest_rel_path = os.path.join(condition, session_folder) + "/"
                
                # === CHECK 1: Source folder has exported_to.json ===
                source_link_file = os.path.join(source_rec_path, "exported_to.json")
                
                if not os.path.exists(source_link_file):
                    stats['missing_source_link'] += 1
                    issues['missing_source_link'].append({
                        'session': session_folder,
                        'source': source_rec_path,
                        'dest': dest_full_path
                    })
                else:
                    # Read and validate source link (NEW FORMAT)
                    with open(source_link_file, 'r') as f:
                        source_data = json.load(f)
                    
                    # Check BOTH full and relative paths
                    full_match = source_data.get('dest_full_path') == dest_full_path
                    rel_match = source_data.get('dest_rel_path') == dest_rel_path
                    
                    if not (full_match and rel_match):
                        stats['source_link_mismatch'] += 1
                        issues['source_mismatch'].append({
                            'session': session_folder,
                            'source': source_rec_path,
                            'expected_full': dest_full_path,
                            'actual_full': source_data.get('dest_full_path'),
                            'expected_rel': dest_rel_path,
                            'actual_rel': source_data.get('dest_rel_path')
                        })
                
                # === CHECK 2: Dest folder_log has export columns ===
                has_full = 'export_dest_full_path' in df.columns
                has_rel = 'export_dest_rel_path' in df.columns
                
                if not (has_full and has_rel):
                    stats['missing_dest_column'] += 1
                    issues['missing_dest_column'].append({
                        'session': session_folder,
                        'dest': dest_full_path,
                        'has_full': has_full,
                        'has_rel': has_rel
                    })
                else:
                    # Validate dest column values
                    full_in_log = df['export_dest_full_path'].iloc[0]
                    rel_in_log = df['export_dest_rel_path'].iloc[0]
                    
                    if full_in_log != dest_full_path or rel_in_log != dest_rel_path:
                        stats['dest_column_mismatch'] += 1
                        issues['dest_mismatch'].append({
                            'session': session_folder,
                            'expected_full': dest_full_path,
                            'actual_full': full_in_log,
                            'expected_rel': dest_rel_path,
                            'actual_rel': rel_in_log
                        })
                
                # === CHECK 3: Compare with master CSV ===
                if source_rec_path in master_mappings:
                    master_dest = master_mappings[source_rec_path]
                    
                    if master_dest.rstrip('/') == dest_rel_path.rstrip('/'):
                        stats['master_csv_match'] += 1
                    else:
                        stats['master_csv_mismatch'] += 1
                        issues['master_mismatch'].append({
                            'session': session_folder,
                            'source': source_rec_path,
                            'master_dest': master_dest,
                            'actual_dest': dest_rel_path
                        })
                else:
                    stats['not_in_master'] += 1
                    issues['not_in_master'].append({
                        'session': session_folder,
                        'source': source_rec_path
                    })
                
                # Count as valid if all checks pass
                if (os.path.exists(source_link_file) and 
                    has_full and has_rel and
                    full_match and rel_match and
                    full_in_log == dest_full_path and 
                    rel_in_log == dest_rel_path):
                    stats['valid_bidirectional'] += 1
                    
            except Exception as e:
                stats['errors'].append((session_folder, str(e)))
    
    return stats, issues
def print_validation_report(dataset_path, stats, issues):
    """Print a detailed validation report"""
    
    print(f"\n{'='*80}")
    print(f"VALIDATION REPORT: {os.path.basename(dataset_path)}")
    print(f"{'='*80}")
    
    # Summary
    print(f"\n📊 SUMMARY:")
    print(f"   Total folders checked: {stats['total_folders']}")
    print(f"   ✅ Valid bidirectional mappings: {stats['valid_bidirectional']}")
    print(f"   ✅ Master CSV matches: {stats['master_csv_match']}")
    print(f"")
    print(f"   ⚠️  Missing source links: {stats['missing_source_link']}")
    print(f"   ⚠️  Missing dest columns: {stats['missing_dest_column']}")
    print(f"   ❌ Source link mismatches: {stats['source_link_mismatch']}")
    print(f"   ❌ Dest column mismatches: {stats['dest_column_mismatch']}")
    print(f"   ❌ Master CSV mismatches: {stats['master_csv_mismatch']}")
    print(f"   ℹ️  Not in master CSV: {stats['not_in_master']}")
    print(f"   ❌ Errors: {len(stats['errors'])}")
    
    # Detailed issues
    if stats['missing_source_link'] > 0:
        print(f"\n⚠️  MISSING SOURCE LINKS (showing first 5):")
        for item in issues['missing_source_link'][:5]:
            print(f"   Session: {item['session']}")
            print(f"   Source: {item['source']}")
            print()
    
    if stats['source_link_mismatch'] > 0:
        print(f"\n❌ SOURCE LINK MISMATCHES (showing first 5):")
        for item in issues['source_mismatch'][:5]:
            print(f"   Session: {item['session']}")
            print(f"   Expected: {item['expected_dest']}")
            print(f"   Actual: {item['actual_dest']}")
            print()
    
    if stats['master_csv_mismatch'] > 0:
        print(f"\n❌ MASTER CSV MISMATCHES (showing first 5):")
        for item in issues['master_mismatch'][:5]:
            print(f"   Session: {item['session']}")
            print(f"   Master: {item['master_dest']}")
            print(f"   Actual: {item['actual_dest']}")
            print()
    
    if len(stats['errors']) > 0:
        print(f"\n❌ ERRORS (showing first 5):")
        for folder, err in stats['errors'][:5]:
            print(f"   {folder}: {err}")
    
    # Calculate health score
    if stats['total_folders'] > 0:
        health_score = (stats['valid_bidirectional'] / stats['total_folders']) * 100
        print(f"\n🏥 HEALTH SCORE: {health_score:.1f}%")

# ============================================================================
# RUN VALIDATION
# ============================================================================

DATASETS = [
    "/data/big_rim/sorted_mir_dataset/oct3v1_single_beh",
    "/data/big_rim/sorted_mir_dataset/oct3v1_single_mini_good",
    "/data/big_rim/sorted_mir_dataset/oct3v1_social_bev",
    "/data/big_rim/sorted_mir_dataset/oct3v1_social_mini"
]

MASTER_CSV = "/home/lq53/mir_repos/BBOP/random_tests/25oct_dataset_prep/export_mapping_master.csv"

print("="*80)
print("VALIDATION: Bidirectional Mapping & Master CSV Check")
print("="*80)

all_results = []

for dataset in DATASETS:
    if os.path.exists(dataset):
        stats, issues = validate_bidirectional_mapping(dataset, MASTER_CSV)
        print_validation_report(dataset, stats, issues)
        all_results.append((dataset, stats, issues))
    else:
        print(f"\n⚠️  Dataset not found: {dataset}")

# Overall summary
print(f"\n{'='*80}")
print("OVERALL SUMMARY")
print(f"{'='*80}")

total_folders = sum(s['total_folders'] for _, s, _ in all_results)
total_valid = sum(s['valid_bidirectional'] for _, s, _ in all_results)
total_issues = sum(
    s['missing_source_link'] + s['missing_dest_column'] + 
    s['source_link_mismatch'] + s['dest_column_mismatch']
    for _, s, _ in all_results
)

print(f"\nAcross all datasets:")
print(f"   Total folders: {total_folders}")
print(f"   Valid bidirectional: {total_valid} ({(total_valid/total_folders*100):.1f}%)")
print(f"   Total issues: {total_issues}")

if total_issues == 0:
    print(f"\n🎉 ALL VALIDATIONS PASSED! 🎉")
else:
    print(f"\n⚠️  Found {total_issues} issues that need attention")

print("="*80)

VALIDATION: Bidirectional Mapping & Master CSV Check

VALIDATING: /data/big_rim/sorted_mir_dataset/oct3v1_single_beh
📖 Loaded master CSV with 121 mappings

VALIDATION REPORT: oct3v1_single_beh

📊 SUMMARY:
   Total folders checked: 48
   ✅ Valid bidirectional mappings: 47
   ✅ Master CSV matches: 47

   ⚠️  Missing source links: 0
   ⚠️  Missing dest columns: 0
   ❌ Source link mismatches: 0
   ❌ Dest column mismatches: 0
   ❌ Master CSV mismatches: 0
   ℹ️  Not in master CSV: 0
   ❌ Errors: 1

❌ ERRORS (showing first 5):
   20251028_144740: No folder_log.parquet

🏥 HEALTH SCORE: 97.9%

VALIDATING: /data/big_rim/sorted_mir_dataset/oct3v1_single_mini_good
📖 Loaded master CSV with 121 mappings

VALIDATION REPORT: oct3v1_single_mini_good

📊 SUMMARY:
   Total folders checked: 39
   ✅ Valid bidirectional mappings: 38
   ✅ Master CSV matches: 38

   ⚠️  Missing source links: 0
   ⚠️  Missing dest columns: 0
   ❌ Source link mismatches: 0
   ❌ Dest column mismatches: 0
   ❌ Master CSV mismatch

In [ ]:
import os
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa

def add_animal_metadata_to_folder_log(dest_folder, session_name):
    """
    Parse session folder name and add animal/session metadata to folder_log.parquet
    
    Args:
        dest_folder: Path to exported session folder
        session_name: Folder name like "20241106_MC7_S1" or "20250227_MC2+MC3_S1"
    
    Example names:
        Single: "20241106_MC7_S1" -> animal1=MC7, animal2=None, session=S1
        Social: "20250227_MC2+MC3_S1" -> animal1=MC2, animal2=MC3, session=S1
    """
    metadata_dir = os.path.join(dest_folder, "metadata")
    log_path = os.path.join(metadata_dir, "folder_log.parquet")
    
    if not os.path.exists(log_path):
        print(f"⚠️  No folder_log.parquet found")
        return False
    
    try:
        # Read existing parquet
        df = pd.read_parquet(log_path)
        
        # Parse session name: YYYYMMDD_ANIMAL(+ANIMAL)_SESSION
        parts = session_name.split('_')
        
        date = None
        animals = []
        session = None
        
        for part in parts:
            # Date (8 digits)
            if part.isdigit() and len(part) == 8:
                date = part
            
            # Animals (contains + or looks like animal ID)
            elif '+' in part:
                animals = part.split('+')
            elif any(part.startswith(prefix) for prefix in ['MC', 'VC', 'PMC', 'V1', 'UNK']):
                animals.append(part)
            
            # Session (S1, S2, etc.)
            elif part.startswith('S') and len(part) > 1 and part[1:].isdigit():
                session = part
        
        # Add new columns
        df['export_date'] = date
        df['animal1'] = animals[0] if len(animals) > 0 else None
        df['animal2'] = animals[1] if len(animals) > 1 else None
        df['session_number'] = session
        df['is_social_export'] = len(animals) > 1
        
        # Save back
        new_table = pa.Table.from_pandas(df)
        pq.write_table(new_table, log_path)
        
        print(f"✅ Added metadata:")
        print(f"   Date: {date}")
        print(f"   Animal1: {animals[0] if len(animals) > 0 else None}")
        print(f"   Animal2: {animals[1] if len(animals) > 1 else None}")
        print(f"   Session: {session}")
        print(f"   Is social: {len(animals) > 1}")
        
        return True
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return False

# ============================================================================
# TEST ON ONE SESSION
# ============================================================================

# Test on a single session
test_folder = "/data/big_rim/sorted_mir_dataset/oct3v1_single_mini_good/single/20241106_MC7_S1"
test_name = "20241106_MC7_S1"

print("="*80)
print("TESTING: Adding animal metadata to folder_log.parquet")
print("="*80)
print(f"\nTest folder: {test_folder}")
print(f"Session name: {test_name}\n")

add_animal_metadata_to_folder_log(test_folder, test_name)

# Verify by reading it back
print("\n" + "="*80)
print("VERIFICATION: Reading back the parquet file")
print("="*80)
log_path = os.path.join(test_folder, "metadata", "folder_log.parquet")
df = pd.read_parquet(log_path)
print("\nNew columns added:")
print(df[['export_date', 'animal1', 'animal2', 'session_number', 'is_social_export']].to_string())

# Test on a social session too
print("\n" + "="*80)
print("TEST 2: Social session")
print("="*80)
test_folder2 = "/data/big_rim/sorted_mir_dataset/oct3v1_social_mini/social/20250227_MC2+MC3_S1"
test_name2 = "20250227_MC2+MC3_S1"

print(f"\nTest folder: {test_folder2}")
print(f"Session name: {test_name2}\n")

add_animal_metadata_to_folder_log(test_folder2, test_name2)

# Verify
log_path2 = os.path.join(test_folder2, "metadata", "folder_log.parquet")
df2 = pd.read_parquet(log_path2)
print("\nNew columns added:")
print(df2[['export_date', 'animal1', 'animal2', 'session_number', 'is_social_export']].to_string())

TESTING: Adding animal metadata to folder_log.parquet

Test folder: /data/big_rim/sorted_mir_dataset/oct3v1_single_mini_good/single/20241106_MC7_S1
Session name: 20241106_MC7_S1

✅ Added metadata:
   Date: 20241106
   Animal1: MC7
   Animal2: None
   Session: S1
   Is social: False

VERIFICATION: Reading back the parquet file

New columns added:
  export_date animal1 animal2 session_number  is_social_export
0    20241106     MC7    None             S1             False

TEST 2: Social session

Test folder: /data/big_rim/sorted_mir_dataset/oct3v1_social_mini/social/20250227_MC2+MC3_S1
Session name: 20250227_MC2+MC3_S1

✅ Added metadata:
   Date: 20250227
   Animal1: MC2
   Animal2: MC3
   Session: S1
   Is social: True

New columns added:
  export_date animal1 animal2 session_number  is_social_export
0    20250227     MC2     MC3             S1              True


In [16]:
import os
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa

def add_animal_metadata_to_folder_log(dest_folder, session_name, skip_if_exists=True):
    """
    Parse session folder name and add animal/session metadata to folder_log.parquet
    """
    metadata_dir = os.path.join(dest_folder, "metadata")
    log_path = os.path.join(metadata_dir, "folder_log.parquet")
    
    if not os.path.exists(log_path):
        return False, "No folder_log.parquet"
    
    try:
        # Read existing parquet
        df = pd.read_parquet(log_path)
        
        # Skip if columns already exist
        if skip_if_exists and 'animal1' in df.columns:
            return False, "Columns already exist"
        
        # Parse session name: YYYYMMDD_ANIMAL(+ANIMAL)_SESSION
        parts = session_name.split('_')
        
        date = None
        animals = []
        session = None
        
        for part in parts:
            # Date (8 digits)
            if part.isdigit() and len(part) == 8:
                date = part
            
            # Animals (contains + or looks like animal ID)
            elif '+' in part:
                animals = part.split('+')
            elif any(part.startswith(prefix) for prefix in ['MC', 'VC', 'PMC', 'V1', 'UNK']):
                animals.append(part)
            
            # Session (S1, S2, etc.)
            elif part.startswith('S') and len(part) > 1 and part[1:].isdigit():
                session = part
        
        # Add new columns - use empty string instead of None for better compatibility
        df['export_date'] = date if date else ""
        df['animal1'] = animals[0] if len(animals) > 0 else ""
        df['animal2'] = animals[1] if len(animals) > 1 else ""  # Empty string instead of None
        df['session_number'] = session if session else ""
        df['is_social_export'] = len(animals) > 1
        
        # Save back
        new_table = pa.Table.from_pandas(df)
        pq.write_table(new_table, log_path)
        
        return True, f"Added: {animals}, {session}"
        
    except Exception as e:
        return False, str(e)
def process_all_sessions(export_root, skip_if_exists=True):
    """
    Process all sessions in a dataset and add animal metadata.
    """
    stats = {
        'processed': 0,
        'added': 0,
        'skipped': 0,
        'errors': []
    }
    
    print(f"\nProcessing: {export_root}")
    print(f"Skip if exists: {skip_if_exists}")
    print("="*80)
    
    for condition in os.listdir(export_root):
        condition_path = os.path.join(export_root, condition)
        if not os.path.isdir(condition_path):
            continue
        
        print(f"\n📁 Condition: {condition}")
        
        for session_folder in os.listdir(condition_path):
            session_path = os.path.join(condition_path, session_folder)
            if not os.path.isdir(session_path):
                continue
            
            stats['processed'] += 1
            
            success, msg = add_animal_metadata_to_folder_log(
                session_path, 
                session_folder, 
                skip_if_exists
            )
            
            if success:
                stats['added'] += 1
                print(f"  ✅ {session_folder}: {msg}")
            else:
                stats['skipped'] += 1
                if "already exist" not in msg:
                    stats['errors'].append((session_folder, msg))
                    print(f"  ⏭️  {session_folder}: {msg}")
    
    return stats

# ============================================================================
# RUN ON ALL DATASETS
# ============================================================================

DATASETS = [
    "/data/big_rim/uploade_ssh_mir_dataset/oct3v1_beh_only",
    "/data/big_rim/uploade_ssh_mir_dataset/oct3v1_social_mini"
    # "/data/big_rim/sorted_mir_dataset/oct3v1_single_beh",
    # "/data/big_rim/sorted_mir_dataset/oct3v1_single_mini_good",
    # "/data/big_rim/sorted_mir_dataset/oct3v1_social_bev",
    # "/data/big_rim/sorted_mir_dataset/oct3v1_social_mini"
]

SKIP_EXISTING = False  # Set to False to UPDATE all

print("="*80)
print("BATCH PROCESSING: Adding animal metadata to folder_log.parquet")
print(f"Mode: {'UPDATE ALL' if not SKIP_EXISTING else 'SKIP EXISTING'}")
print("="*80)

all_stats = []
for dataset in DATASETS:
    if os.path.exists(dataset):
        stats = process_all_sessions(dataset, skip_if_exists=SKIP_EXISTING)
        all_stats.append((dataset, stats))
    else:
        print(f"\n⚠️  Dataset not found: {dataset}")

# Print summary
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
for dataset, stats in all_stats:
    print(f"\n📁 {os.path.basename(dataset)}")
    print(f"   Processed: {stats['processed']} folders")
    print(f"   Added metadata: {stats['added']}")
    print(f"   Skipped: {stats['skipped']}")
    print(f"   Errors: {len(stats['errors'])}")
    
    if stats['errors']:
        print(f"\n   ❌ Errors (first 5):")
        for folder, err in stats['errors'][:5]:
            print(f"      {folder}: {err}")

total_added = sum(s['added'] for _, s in all_stats)
total_processed = sum(s['processed'] for _, s in all_stats)

print(f"\n{'='*80}")
print(f"TOTAL: {total_added}/{total_processed} folders updated")
print("="*80)
print("✅ Done! All folder_log.parquet files now have animal metadata")

BATCH PROCESSING: Adding animal metadata to folder_log.parquet
Mode: UPDATE ALL

Processing: /data/big_rim/uploade_ssh_mir_dataset/oct3v1_beh_only
Skip if exists: False

📁 Condition: .trash
  ⏭️  20251028_132918: No folder_log.parquet

📁 Condition: single
  ✅ 20241113_VC3_S1: Added: ['VC3'], S1
  ✅ 20241004_VC4_S5: Added: ['VC4'], S5
  ✅ 20240918_MC14_S1: Added: ['MC14'], S1
  ✅ 20241106_MC17_S1: Added: ['MC17'], S1
  ✅ 20241007_VC10_S3: Added: ['VC10'], S3
  ✅ 20250521_VC12_S1: Added: ['VC12'], S1
  ✅ 20250213_MC16_S2: Added: ['MC16'], S2
  ✅ 20250514_MC12_S2: Added: ['MC12'], S2
  ✅ 20241008_VC10_S7: Added: ['VC10'], S7
  ✅ 20240918_VC4_S2: Added: ['VC4'], S2
  ✅ 20241024_MC15_S1: Added: ['MC15'], S1
  ✅ 20241014_VC10_S9: Added: ['VC10'], S9
  ✅ 20241106_UNK17_S1: Added: ['UNK17'], S1
  ✅ 20240918_VC4_S1: Added: ['VC4'], S1
  ✅ 20241008_VC10_S5: Added: ['VC10'], S5
  ✅ 20241024_MC1_S1: Added: ['MC1'], S1
  ✅ 20241008_VC10_S8: Added: ['VC10'], S8
  ✅ 20250410_VC14_S1: Added: ['VC14'],

In [12]:
import os
import pandas as pd

def validate_animal_metadata(export_root):
    """
    Validate that all folder_log.parquet files have correct animal metadata.
    Check that parsed metadata matches the folder name.
    """
    
    print(f"\n{'='*80}")
    print(f"VALIDATING ANIMAL METADATA: {export_root}")
    print(f"{'='*80}")
    
    stats = {
        'total_folders': 0,
        'valid': 0,
        'missing_columns': 0,
        'parsing_errors': 0,
        'mismatches': 0,
        'errors': []
    }
    
    issues = {
        'missing_columns': [],
        'parsing_errors': [],
        'mismatches': []
    }
    
    for condition in os.listdir(export_root):
        condition_path = os.path.join(export_root, condition)
        if not os.path.isdir(condition_path):
            continue
        
        for session_folder in os.listdir(condition_path):
            session_path = os.path.join(condition_path, session_folder)
            if not os.path.isdir(session_path):
                continue
            
            stats['total_folders'] += 1
            
            log_path = os.path.join(session_path, "metadata", "folder_log.parquet")
            
            if not os.path.exists(log_path):
                stats['errors'].append((session_folder, "No folder_log.parquet"))
                continue
            
            try:
                df = pd.read_parquet(log_path)
                
                # Check if required columns exist
                required_cols = ['export_date', 'animal1', 'animal2', 'session_number', 'is_social_export']
                missing = [col for col in required_cols if col not in df.columns]
                
                if missing:
                    stats['missing_columns'] += 1
                    issues['missing_columns'].append({
                        'session': session_folder,
                        'missing': missing
                    })
                    continue
                
                # Parse expected values from folder name
                parts = session_folder.split('_')
                
                expected_date = None
                expected_animals = []
                expected_session = None
                
                for part in parts:
                    if part.isdigit() and len(part) == 8:
                        expected_date = part
                    elif '+' in part:
                        expected_animals = part.split('+')
                    elif any(part.startswith(prefix) for prefix in ['MC', 'VC', 'PMC', 'V1', 'UNK']):
                        expected_animals.append(part)
                    elif part.startswith('S') and len(part) > 1 and part[1:].isdigit():
                        expected_session = part
                
                # Validate parsed values match stored values
                actual_date = df['export_date'].iloc[0]
                actual_animal1 = df['animal1'].iloc[0]
                actual_animal2 = df['animal2'].iloc[0]
                actual_session = df['session_number'].iloc[0]
                actual_is_social = df['is_social_export'].iloc[0]
                
                expected_animal1 = expected_animals[0] if len(expected_animals) > 0 else None
                expected_animal2 = expected_animals[1] if len(expected_animals) > 1 else None
                expected_is_social = len(expected_animals) > 1
                
                # Check for mismatches
                mismatches = []
                if actual_date != expected_date:
                    mismatches.append(f"date: {actual_date} != {expected_date}")
                if actual_animal1 != expected_animal1:
                    mismatches.append(f"animal1: {actual_animal1} != {expected_animal1}")
                if actual_animal2 != expected_animal2:
                    mismatches.append(f"animal2: {actual_animal2} != {expected_animal2}")
                if actual_session != expected_session:
                    mismatches.append(f"session: {actual_session} != {expected_session}")
                if actual_is_social != expected_is_social:
                    mismatches.append(f"is_social: {actual_is_social} != {expected_is_social}")
                
                if mismatches:
                    stats['mismatches'] += 1
                    issues['mismatches'].append({
                        'session': session_folder,
                        'mismatches': mismatches
                    })
                else:
                    stats['valid'] += 1
                    
            except Exception as e:
                stats['parsing_errors'] += 1
                issues['parsing_errors'].append({
                    'session': session_folder,
                    'error': str(e)
                })
    
    return stats, issues

def print_validation_report(dataset_path, stats, issues):
    """Print validation report for animal metadata"""
    
    print(f"\n{'='*80}")
    print(f"VALIDATION REPORT: {os.path.basename(dataset_path)}")
    print(f"{'='*80}")
    
    print(f"\n📊 SUMMARY:")
    print(f"   Total folders: {stats['total_folders']}")
    print(f"   ✅ Valid metadata: {stats['valid']}")
    print(f"   ⚠️  Missing columns: {stats['missing_columns']}")
    print(f"   ❌ Parsing errors: {stats['parsing_errors']}")
    print(f"   ❌ Mismatches: {stats['mismatches']}")
    print(f"   ❌ Other errors: {len(stats['errors'])}")
    
    # Show issues
    if stats['missing_columns'] > 0:
        print(f"\n⚠️  MISSING COLUMNS (first 5):")
        for item in issues['missing_columns'][:5]:
            print(f"   {item['session']}: missing {item['missing']}")
    
    if stats['mismatches'] > 0:
        print(f"\n❌ METADATA MISMATCHES (first 5):")
        for item in issues['mismatches'][:5]:
            print(f"   {item['session']}:")
            for mismatch in item['mismatches']:
                print(f"      - {mismatch}")
    
    if stats['parsing_errors'] > 0:
        print(f"\n❌ PARSING ERRORS (first 5):")
        for item in issues['parsing_errors'][:5]:
            print(f"   {item['session']}: {item['error']}")
    
    if len(stats['errors']) > 0:
        print(f"\n❌ OTHER ERRORS (first 5):")
        for folder, err in stats['errors'][:5]:
            print(f"   {folder}: {err}")
    
    # Health score
    if stats['total_folders'] > 0:
        health = (stats['valid'] / stats['total_folders']) * 100
        print(f"\n🏥 HEALTH SCORE: {health:.1f}%")

# ============================================================================
# RUN VALIDATION
# ============================================================================

DATASETS = [
    # "/data/big_rim/sorted_mir_dataset/oct3v1_single_beh",
    # "/data/big_rim/sorted_mir_dataset/oct3v1_single_mini_good",
    # "/data/big_rim/sorted_mir_dataset/oct3v1_social_bev",
    # "/data/big_rim/sorted_mir_dataset/oct3v1_social_mini"
    "/data/big_rim/uploade_ssh_mir_dataset/oct3v1_beh_only",
    "/data/big_rim/uploade_ssh_mir_dataset/oct3v1_social_mini"
]

print("="*80)
print("VALIDATION: Animal Metadata in folder_log.parquet")
print("="*80)

all_results = []
for dataset in DATASETS:
    if os.path.exists(dataset):
        stats, issues = validate_animal_metadata(dataset)
        print_validation_report(dataset, stats, issues)
        all_results.append((dataset, stats, issues))
    else:
        print(f"\n⚠️  Dataset not found: {dataset}")

# Overall summary
print(f"\n{'='*80}")
print("OVERALL SUMMARY")
print(f"{'='*80}")

total_folders = sum(s['total_folders'] for _, s, _ in all_results)
total_valid = sum(s['valid'] for _, s, _ in all_results)
total_issues = sum(
    s['missing_columns'] + s['parsing_errors'] + s['mismatches']
    for _, s, _ in all_results
)

print(f"\nAcross all datasets:")
print(f"   Total folders: {total_folders}")
print(f"   Valid metadata: {total_valid} ({(total_valid/total_folders*100):.1f}%)")
print(f"   Total issues: {total_issues}")

if total_issues == 0:
    print(f"\n🎉 ALL VALIDATIONS PASSED! 🎉")
else:
    print(f"\n⚠️  Found {total_issues} issues that need attention")

print("="*80)

VALIDATION: Animal Metadata in folder_log.parquet

VALIDATING ANIMAL METADATA: /data/big_rim/uploade_ssh_mir_dataset/oct3v1_beh_only

VALIDATION REPORT: oct3v1_beh_only

📊 SUMMARY:
   Total folders: 73
   ✅ Valid metadata: 72
   ⚠️  Missing columns: 0
   ❌ Parsing errors: 0
   ❌ Mismatches: 0
   ❌ Other errors: 1

❌ OTHER ERRORS (first 5):
   20251028_132918: No folder_log.parquet

🏥 HEALTH SCORE: 98.6%

VALIDATING ANIMAL METADATA: /data/big_rim/uploade_ssh_mir_dataset/oct3v1_social_mini

VALIDATION REPORT: oct3v1_social_mini

📊 SUMMARY:
   Total folders: 51
   ✅ Valid metadata: 49
   ⚠️  Missing columns: 0
   ❌ Parsing errors: 0
   ❌ Mismatches: 0
   ❌ Other errors: 2

❌ OTHER ERRORS (first 5):
   20251027_170409: No folder_log.parquet
   20251027_170226: No folder_log.parquet

🏥 HEALTH SCORE: 96.1%

OVERALL SUMMARY

Across all datasets:
   Total folders: 124
   Valid metadata: 121 (97.6%)
   Total issues: 0

🎉 ALL VALIDATIONS PASSED! 🎉


In [25]:
import sys, os
sys.path.append(os.path.abspath('../..'))
from utlis.scan_engine_utlis.scan_engine_utlis import read_all_parquet_files

base_folder = "/data/big_rim/uploade_ssh_mir_dataset"
all_df = read_all_parquet_files(base_folder)

In [31]:
all_df.to_pandas()

,mir_generate_param,sync,mini_6cam_map,dropf_handle,com,com_vis,social,miniscope,test,after_oxytocin,...,date_folder,calib_files,export_dest_path,export_dest_full_path,export_dest_rel_path,export_date,animal1,animal2,session_number,is_social_export
0,1,1,1,0,1,1,0,1,0,0,...,2024_10_24,"[calib_14_28, calib_before_newintrinsics, cali...",/data/big_rim/sorted_mir_dataset/oct3v1_single...,/data/big_rim/sorted_mir_dataset/oct3v1_single...,single/20241024_MC8_S1/,20241024,MC8,,S1,False
1,1,1,1,0,1,1,0,1,0,0,...,2024_10_17,"[calib_before_newintrinsics, calib_before]",/data/big_rim/sorted_mir_dataset/oct3v1_single...,/data/big_rim/sorted_mir_dataset/oct3v1_single...,single/20241017_VC4_S3/,20241017,VC4,,S3,False
2,1,1,1,0,1,1,0,1,0,1,...,2024_11_07,"[calib_15_39, calib_before_newintrinsics, cali...",/data/big_rim/sorted_mir_dataset/oct3v1_single...,/data/big_rim/sorted_mir_dataset/oct3v1_single...,single/20241107_MC7_S3/,20241107,MC7,,S3,False
3,1,1,1,0,1,1,0,1,0,0,...,2025_02_12,"[calib_after, calib_before_newintrinsics, cali...",/data/big_rim/sorted_mir_dataset/oct3v1_single...,/data/big_rim/sorted_mir_dataset/oct3v1_single...,single/20250212_MC8_S3/,20250212,MC8,,S3,False
4,1,1,1,0,1,1,0,1,0,0,...,2025_02_12,"[calib_after, calib_before_newintrinsics, cali...",/data/big_rim/sorted_mir_dataset/oct3v1_single...,/data/big_rim/sorted_mir_dataset/oct3v1_single...,single/20250212_MC8_S5/,20250212,MC8,,S5,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
237,1,1,1,0,1,1,1,1,0,0,...,2025_05_14,"[calib_after, calib_before_newintrinsics, cali...",/data/big_rim/sorted_mir_dataset/oct3v1_social...,/data/big_rim/uploade_ssh_mir_dataset/oct3v1_b...,oct3v1_beh_only/social/20250514_MC13+UNK16_S1/,20250514,MC13,UNK16,S1,True
238,1,1,1,0,1,1,1,1,0,0,...,2024_12_31,"[calib_before_newintrinsics, calib_before]",/data/big_rim/sorted_mir_dataset/oct3v1_social...,/data/big_rim/uploade_ssh_mir_dataset/oct3v1_b...,oct3v1_beh_only/social/20241231_VC1+MC1_S1/,20241231,VC1,MC1,S1,True
239,1,1,0,0,1,1,1,1,0,0,...,2024_10_30,"[calib_before_newintrinsics, calib_before]",/data/big_rim/sorted_mir_dataset/oct3v1_social...,/data/big_rim/uploade_ssh_mir_dataset/oct3v1_b...,oct3v1_beh_only/social/20241030_MC10+UNK1_S6/,20241030,MC10,UNK1,S6,True
240,1,1,1,0,1,1,1,1,0,0,...,2024_12_31,"[calib_before_newintrinsics, calib_before]",/data/big_rim/sorted_mir_dataset/oct3v1_social...,/data/big_rim/uploade_ssh_mir_dataset/oct3v1_b...,oct3v1_beh_only/social/20241231_VC6+UNK12_S1/,20241231,VC6,UNK12,S1,True


In [27]:
import pyarrow.parquet as pq

# Save as parquet (PyArrow Table)
pq.write_table(all_df, "dataset_master_metadata.parquet")

# Save as CSV (convert to pandas first)
all_df.to_pandas().to_csv("dataset_master_metadata.csv", index=False)

In [32]:
import pyarrow.compute as pc
from functools import reduce


table = all_df #combined_df
# Filter mir_generate_param == 0 and sync != 3
# ============================================================================
# FILTERING CONDITIONS
# ============================================================================

# Define your filtering criteria
conditions = [
    pc.equal(table['sync'], '1'),        # Videos are synchronized
    pc.equal(table['com'], '1'),         # Center of mass tracking complete
    # pc.equal(table['dannce'], '1'),    # 3D pose estimation complete
    # pc.equal(table['social'], '0'),    # Solo recordings only (0=solo, 1=social)
    # pc.equal(table['mini_6cam_map'], '1'),  # 6-camera miniscope mapping done
]


filter_mask = reduce(pc.and_, conditions)


# Apply the filter and print the results
filtered_table = table.filter(filter_mask)

# ============================================================================
# DISPLAY RESULTS
# ============================================================================

# Convert to pandas for easier viewing
filtered_df = filtered_table.to_pandas()

# Select meaningful columns for display
display_cols = ['export_dest_rel_path', 'export_dest_full_path', 'social', 'test', 'before_oxytocin', 'after_oxytocin']

print(f"\nFiltered: {len(filtered_df)} recordings")
filtered_df[display_cols]


Filtered: 242 recordings


,export_dest_rel_path,export_dest_full_path,social,test,before_oxytocin,after_oxytocin
0,single/20241024_MC8_S1/,/data/big_rim/sorted_mir_dataset/oct3v1_single...,0,0,0,0
1,single/20241017_VC4_S3/,/data/big_rim/sorted_mir_dataset/oct3v1_single...,0,0,0,0
2,single/20241107_MC7_S3/,/data/big_rim/sorted_mir_dataset/oct3v1_single...,0,0,0,1
3,single/20250212_MC8_S3/,/data/big_rim/sorted_mir_dataset/oct3v1_single...,0,0,0,0
4,single/20250212_MC8_S5/,/data/big_rim/sorted_mir_dataset/oct3v1_single...,0,0,0,0
...,...,...,...,...,...,...
237,oct3v1_beh_only/social/20250514_MC13+UNK16_S1/,/data/big_rim/uploade_ssh_mir_dataset/oct3v1_b...,1,0,0,0
238,oct3v1_beh_only/social/20241231_VC1+MC1_S1/,/data/big_rim/uploade_ssh_mir_dataset/oct3v1_b...,1,0,0,0
239,oct3v1_beh_only/social/20241030_MC10+UNK1_S6/,/data/big_rim/uploade_ssh_mir_dataset/oct3v1_b...,1,0,0,0
240,oct3v1_beh_only/social/20241231_VC6+UNK12_S1/,/data/big_rim/uploade_ssh_mir_dataset/oct3v1_b...,1,0,0,0
